# 02 · Prompt development on a small $WIF sample (exploratory)

Before scoring the full Dogecoin sample, I tested approaches on ~200 posts around a Dogwifhat ($WIF) event:
1. **FinBERT baseline** (`yiyanghkust/finbert-tone`) — positive / neutral / negative only.
2. **LLM JSON scoring** with DeepSeek — richer labels returned as structured JSON.
3. **Fact / opinion / question step** — early prompts gave emotion scores to plain statements like "price is $0.08", so the prompt was changed to classify the type of post first and only score stance and emotion for opinions.

This notebook is development work and is **not** part of the paper's results. Printed outputs that showed individual users' posts have been cleared.

BERT

In [ ]:
# pip install transformers torch pandas openpyxl

import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# === Step 2: 加载数据 ===
df = pd.read_excel("../data/raw/dogwifhat_sphere_event_optimized_20251013_012550.xlsx")

# 确保有文本字段（假设列名为 "text" 或 "comment"）
if "text" not in df.columns:
    text_col = [c for c in df.columns if "text" in c.lower() or "comment" in c.lower()]
    if text_col:
        df.rename(columns={text_col[0]: "text"}, inplace=True)
    else:
        raise ValueError("找不到评论文本列，请检查文件列名。")

# 取非空文本
texts = df["text"].dropna().astype(str).tolist()

# === Step 3: 加载 FinBERT 模型 ===
MODEL_NAME = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# === Step 4: 逐条预测 ===
labels = ["positive", "neutral", "negative"]

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
    return labels[pred], confidence

results = [predict_sentiment(t) for t in texts]

# === Step 5: 结果整合 ===
df["sentiment_label"] = [r[0] for r in results]
df["confidence"] = [r[1] for r in results]

# 统计整体情绪比例
sentiment_summary = df["sentiment_label"].value_counts(normalize=True) * 100
print("Sentiment Distribution (%):\n", sentiment_summary)

# === Step 6: 导出结果 ===
df.to_excel("dogwifhat_sphere_event_with_sentiment.xlsx", index=False)
print("已保存情绪分析结果 -> dogwifhat_sphere_event_with_sentiment.xlsx")


vocab.txt: 0.00B [00:00, ?B/s]

Sentiment Distribution (%):
 sentiment_label
positive    86.069652
negative     7.960199
neutral      5.970149
Name: proportion, dtype: float64
已保存情绪分析结果 -> dogwifhat_sphere_event_with_sentiment.xlsx


First Try

In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json

# === 0. 配置 API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# === 1. 读取 Excel 数据 ===
df = pd.read_excel("../data/raw/dogwifhat_sphere_event_optimized_20251013_012550.xlsx")

# 修改为你实际的文本列名
TEXT_COLUMN = "text"

# === 2. 定义情绪分析函数 ===
def analyze_sentiment(text):
    prompt = f"""
    Please analyze the following social media comment about the Dogwifhat Las Vegas Sphere advertisement event.
    Use behavioral finance and social psychology perspective.

    Classify each comment according to:
    1. stance ∈ ["Promotion", "Support", "Neutral", "Question", "Opposition"]
    2. valence ∈ ["Positive", "Neutral", "Negative"]
    3. arousal ∈ ["Low", "Medium", "High"]
    4. reason: one-sentence rationale

    Comment: {text}

    Respond in valid JSON format like:
    {{
      "stance": "...",
      "valence": "...",
      "arousal": "...",
      "reason": "..."
    }}
    """

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are an expert financial sentiment analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
        )
        result_text = response.choices[0].message.content.strip()
        # 尝试解析 JSON（防止模型输出文本格式错误）
        try:
            result_json = json.loads(result_text)
        except json.JSONDecodeError:
            result_json = {"raw_output": result_text}
        return result_json
    except Exception as e:
        print("Error:", e)
        return {"error": str(e)}

# === 3. 批量分析 ===
results = []
for i, row in df.iterrows():
    text = str(row[TEXT_COLUMN])
    if not text.strip():
        results.append({"stance": None, "valence": None, "arousal": None, "reason": None})
        continue

    print(f"Analyzing {i+1}/{len(df)}: {text[:50]}...")
    output = analyze_sentiment(text)

    # 标准化输出
    stance = output.get("stance") if isinstance(output, dict) else None
    valence = output.get("valence") if isinstance(output, dict) else None
    arousal = output.get("arousal") if isinstance(output, dict) else None
    reason = output.get("reason") if isinstance(output, dict) else output

    results.append({
        "stance": stance,
        "valence": valence,
        "arousal": arousal,
        "reason": reason
    })

    time.sleep(1.5)  # 防止速率限制

# === 4. 保存结果 ===
result_df = pd.concat([df, pd.DataFrame(results)], axis=1)
output_path = "dogwifhat_sentiment_results.xlsx"
result_df.to_excel(output_path, index=False)

print(f"✅ Sentiment analysis completed and saved to {output_path}")


區分新聞

In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json
import re

# === 配置 API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# === 数据读取和清洗 ===
df = pd.read_excel("../data/raw/dogwifhat_sphere_event_optimized_20251013_012550.xlsx")

def clean_text(text):
    """彻底清洗文本数据"""
    if pd.isna(text) or text is None:
        return None
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text if text and text != 'nan' else None

df['cleaned_text'] = df['text'].apply(clean_text)

# === 测试前5条有效数据 ===
print("🔍 测试前5条有效数据...")
print("=" * 60)

test_count = 0
results = []

for i, row in df.iterrows():
    text = row['cleaned_text']

    if not text:
        continue

    test_count += 1
    if test_count > 5:  # 只测试5条
        break

    print(f"\n📝 测试 {test_count}:")
    print(f"原文: {text}")

    # 使用改进的分析函数
    prompt = f"""
You are a precise financial sentiment annotator.

Analyze the following social media comment about the Dogwifhat Las Vegas Sphere advertising controversy.
Output **strictly valid JSON**, no explanations, no markdown, no code block.

Comment: {text}

Label according to these rules:
- stance ∈ ["Promotion", "Support", "Neutral", "Question", "Opposition"]
- valence ∈ ["Positive", "Neutral", "Negative"]
- arousal ∈ ["Low", "Medium", "High"]
- reason: 1 short sentence explaining the reasoning.

Output format (one single JSON object):
{{"stance": "...", "valence": "...", "arousal": "...", "reason": "..."}}
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are an expert financial sentiment analyst. Always output valid JSON format."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        result_text = response.choices[0].message.content.strip()
        result_json = json.loads(result_text)

        print(f"✅ 分析结果:")
        print(f"   立场: {result_json.get('stance')}")
        print(f"   情感: {result_json.get('valence')}")
        print(f"   强度: {result_json.get('arousal')}")
        print(f"   理由: {result_json.get('reason')}")

        results.append(result_json)

    except Exception as e:
        print(f"❌ 分析失败: {e}")
        results.append({"error": str(e)})

    time.sleep(1.2)

print("\n" + "=" * 60)
print(f"🎯 测试完成! 成功分析 {len([r for r in results if 'error' not in r])}/5 条数据")

# 显示统计信息
if results:
    stances = [r.get('stance', 'Unknown') for r in results if 'error' not in r]
    valence = [r.get('valence', 'Unknown') for r in results if 'error' not in r]

    print(f"\n📊 测试统计:")
    print(f"立场分布: {pd.Series(stances).value_counts().to_dict()}")
    print(f"情感分布: {pd.Series(valence).value_counts().to_dict()}")

小範圍測試，情感

In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json
import re

# === 配置 API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# === 数据读取和清洗 ===
df = pd.read_excel("../data/raw/dogwifhat_sphere_event_optimized_20251013_012550.xlsx")

def clean_text(text):
    """彻底清洗文本数据"""
    if pd.isna(text) or text is None:
        return None
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text if text and text != 'nan' else None

df['cleaned_text'] = df['text'].apply(clean_text)

# === 修正的情绪分析函数 - 明确站在WIF立场 ===
def analyze_sentiment_wif_perspective(text):
    """更精确的WIF立场分析 - 区分事实陈述和情感表达"""
    prompt = f"""
You are a precise financial sentiment annotator. Analyze the sentiment specifically towards Dogwifhat ($WIF) in the following comment.

Comment: {text}

IMPORTANT:
1. Focus ONLY on sentiment towards Dogwifhat ($WIF)
2. Distinguish between factual statements and emotional opinions
3. If the comment is purely factual/informational without emotional bias, classify as Neutral

Label according to these rules:
- stance ∈ ["Promotion", "Support", "Neutral", "Question", "Opposition"] - specifically towards WIF
- valence ∈ ["Positive", "Neutral", "Negative"] - specifically towards WIF
- arousal ∈ ["Low", "Medium", "High"] - emotional intensity towards WIF
- reason: 1 short sentence explaining the classification

CRITICAL GUIDELINES:
- "Neutral" stance: for factual reports, probability statements, news sharing without opinion
- "Question" stance: for inquiries seeking information
- "Support/Opposition": must contain clear emotional bias or opinion

Output format (strictly valid JSON):
{{"stance": "...", "valence": "...", "arousal": "...", "reason": "..."}}
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are an expert financial sentiment analyst. Always analyze sentiment specifically towards Dogwifhat (WIF). Ignore other tokens. Always output valid JSON format."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        result_text = response.choices[0].message.content.strip()
        result_json = json.loads(result_text)
        return result_json

    except Exception as e:
        print(f"❌ API调用错误: {e}")
        return {"stance": "Error", "valence": "Error", "arousal": "Error", "reason": f"API Error: {str(e)}"}

# === 测试修正后的分析 ===
print("🔍 测试修正后的WIF立场分析...")
print("=" * 60)

test_count = 0
results = []

for i, row in df.iterrows():
    text = row['cleaned_text']

    if not text:
        continue

    test_count += 1
    if test_count > 5:  # 只测试5条
        break

    print(f"\n📝 测试 {test_count}:")
    print(f"原文: {text}")

    result = analyze_sentiment_wif_perspective(text)

    print(f"✅ WIF立场分析结果:")
    print(f"   立场: {result.get('stance')}")
    print(f"   情感: {result.get('valence')}")
    print(f"   强度: {result.get('arousal')}")
    print(f"   理由: {result.get('reason')}")

    results.append(result)
    time.sleep(1.2)

print("\n" + "=" * 60)
print(f"🎯 修正测试完成! 成功分析 {len([r for r in results if 'error' not in r])}/5 条数据")

# 显示WIF立场统计
if results:
    stances = [r.get('stance', 'Unknown') for r in results if 'error' not in r]
    valence = [r.get('valence', 'Unknown') for r in results if 'error' not in r]

    print(f"\n📊 WIF立场统计:")
    print(f"立场分布: {pd.Series(stances).value_counts().to_dict()}")
    print(f"情感分布: {pd.Series(valence).value_counts().to_dict()}")

# === 完整批量分析函数 ===
def batch_analyze_wif_sentiment(df, start_index=0, end_index=None):
    """批量分析对WIF的情绪"""
    if end_index is None:
        end_index = len(df)

    results = []
    valid_texts = df['cleaned_text'].dropna()

    print(f"🚀 开始批量分析WIF立场情绪...")
    print(f"📊 总数据量: {len(valid_texts)}")

    for i, row in df.iterrows():
        if i < start_index:
            continue
        if i >= end_index:
            break

        text = row['cleaned_text']
        if not text:
            results.append({"stance": "Empty", "valence": "Empty", "arousal": "Empty", "reason": "Empty text"})
            continue

        print(f"🔍 分析 {i+1}/{len(df)}: {text[:80]}...")

        try:
            output = analyze_sentiment_wif_perspective(text)
            results.append(output)
            print(f"✅ WIF立场: {output.get('stance', 'N/A')} | 情感: {output.get('valence', 'N/A')}")

        except Exception as e:
            print(f"❌ 分析失败 {i+1}: {e}")
            results.append({"stance": "Error", "valence": "Error", "arousal": "Error", "reason": str(e)})

        time.sleep(1.2)

    return results

# 如果需要运行完整分析，取消注释下面这行
final_results = batch_analyze_wif_sentiment(df)

改進版DeepSeek指令分析 (biased)

In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json
import re

# === 配置 API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# === 数据读取和清洗 ===
df = pd.read_excel("../data/raw/RAW_dogwifhat.xlsx")

def clean_text(text):
    """彻底清洗文本数据"""
    if pd.isna(text) or text is None:
        return None
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text if text and text != 'nan' else None

df['cleaned_text'] = df['text'].apply(clean_text)

# === 修正的情绪分析函数 - 明确站在WIF立场 ===
def analyze_sentiment_wif_perspective(text):
    """更精确的WIF立场分析 - 区分事实陈述和情感表达"""
    prompt = f"""
You are a precise financial sentiment annotator. Analyze the sentiment specifically towards Dogwifhat ($WIF) in the following comment.

Comment: {text}

IMPORTANT:
1. Focus ONLY on sentiment towards Dogwifhat ($WIF)
2. Distinguish between factual statements and emotional opinions
3. If the comment is purely factual/informational without emotional bias, classify as Neutral

Label according to these rules:
- stance ∈ ["Promotion", "Support", "Neutral", "Question", "Opposition"] - specifically towards WIF
- valence ∈ ["Positive", "Neutral", "Negative"] - specifically towards WIF
- arousal ∈ ["Low", "Medium", "High"] - emotional intensity towards WIF
- reason: 1 short sentence explaining the classification

CRITICAL GUIDELINES:
- "Neutral" stance: for factual reports, probability statements, news sharing without opinion
- "Question" stance: for inquiries seeking information
- "Support/Opposition": must contain clear emotional bias or opinion

Output format (strictly valid JSON):
{{"stance": "...", "valence": "...", "arousal": "...", "reason": "..."}}
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are an expert financial sentiment analyst. Always analyze sentiment specifically towards Dogwifhat (WIF). Ignore other tokens. Always output valid JSON format."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        result_text = response.choices[0].message.content.strip()
        result_json = json.loads(result_text)
        return result_json

    except Exception as e:
        print(f"❌ API调用错误: {e}")
        return {"stance": "Error", "valence": "Error", "arousal": "Error", "reason": f"API Error: {str(e)}"}

# === 完整批量分析函数 ===
def batch_analyze_all_data(df):
    """批量分析全部数据并保存结果"""
    results = []
    total_rows = len(df)
    valid_count = 0

    print(f"🚀 开始批量分析全部数据...")
    print(f"📊 总数据量: {total_rows}")
    print(f"⏰ 预计时间: {total_rows * 1.5 / 60:.1f} 分钟")

    for i, row in df.iterrows():
        text = row['cleaned_text']

        # 进度显示
        if (i + 1) % 10 == 0:
            print(f"📈 进度: {i+1}/{total_rows} ({((i+1)/total_rows*100):.1f}%)")

        if not text:
            results.append({
                "stance": "Empty",
                "valence": "Empty",
                "arousal": "Empty",
                "reason": "Empty text",
                "analysis_status": "Skipped"
            })
            continue

        try:
            output = analyze_sentiment_wif_perspective(text)
            output["analysis_status"] = "Success"
            results.append(output)
            valid_count += 1

        except Exception as e:
            print(f"❌ 第 {i+1} 行分析失败: {e}")
            results.append({
                "stance": "Error",
                "valence": "Error",
                "arousal": "Error",
                "reason": str(e),
                "analysis_status": "Failed"
            })

        time.sleep(1.2)  # 防止速率限制

    return results, valid_count

# === 执行批量分析 ===
print("=" * 60)
print("🎯 开始执行全部数据情感分析")
print("=" * 60)

start_time = time.time()
final_results, valid_count = batch_analyze_all_data(df)
end_time = time.time()

# === 合并结果并保存 ===
print("\n💾 正在保存分析结果...")

# 将分析结果转换为DataFrame
results_df = pd.DataFrame(final_results)

# 合并原始数据和分析结果
final_output_df = pd.concat([df, results_df], axis=1)

# 保存到Excel文件
output_filename = "DS_v2_dogwifhat_sentiment_analysis.xlsx"
final_output_df.to_excel(output_filename, index=False)

# === 显示分析统计 ===
analysis_time = end_time - start_time
total_analyzed = len([r for r in final_results if r.get('analysis_status') == 'Success'])

print("=" * 60)
print("📊 分析完成统计:")
print(f"✅ 总数据行数: {len(df)}")
print(f"✅ 成功分析: {total_analyzed} 条")
print(f"✅ 空文本跳过: {len([r for r in final_results if r.get('analysis_status') == 'Skipped'])} 条")
print(f"✅ 分析失败: {len([r for r in final_results if r.get('analysis_status') == 'Failed'])} 条")
print(f"⏱️ 总耗时: {analysis_time/60:.1f} 分钟")
print(f"💾 结果文件: {output_filename}")
print("=" * 60)

# 显示情感分布统计
successful_results = [r for r in final_results if r.get('analysis_status') == 'Success']
if successful_results:
    stances = [r.get('stance', 'Unknown') for r in successful_results]
    valence = [r.get('valence', 'Unknown') for r in successful_results]

    print("\n🎯 情感分析统计:")
    print(f"立场分布:")
    for stance, count in pd.Series(stances).value_counts().items():
        print(f"  {stance}: {count} 条 ({count/len(successful_results)*100:.1f}%)")

    print(f"\n情感分布:")
    for val, count in pd.Series(valence).value_counts().items():
        print(f"  {val}: {count} 条 ({count/len(successful_results)*100:.1f}%)")

print(f"\n🎉 全部数据分析完成！结果已保存至: {output_filename}")

🎯 开始执行全部数据情感分析
🚀 开始批量分析全部数据...
📊 总数据量: 201
⏰ 预计时间: 5.0 分钟
📈 进度: 10/201 (5.0%)
📈 进度: 20/201 (10.0%)
📈 进度: 30/201 (14.9%)
📈 进度: 40/201 (19.9%)
📈 进度: 50/201 (24.9%)
📈 进度: 60/201 (29.9%)
📈 进度: 70/201 (34.8%)
📈 进度: 80/201 (39.8%)
📈 进度: 90/201 (44.8%)
📈 进度: 100/201 (49.8%)
📈 进度: 110/201 (54.7%)
📈 进度: 120/201 (59.7%)
📈 进度: 130/201 (64.7%)
📈 进度: 140/201 (69.7%)
📈 进度: 150/201 (74.6%)
📈 进度: 160/201 (79.6%)
📈 进度: 170/201 (84.6%)
📈 进度: 180/201 (89.6%)
📈 进度: 190/201 (94.5%)
📈 进度: 200/201 (99.5%)

💾 正在保存分析结果...
📊 分析完成统计:
✅ 总数据行数: 201
✅ 成功分析: 201 条
✅ 空文本跳过: 0 条
✅ 分析失败: 0 条
⏱️ 总耗时: 16.7 分钟
💾 结果文件: DS_v2_dogwifhat_sentiment_analysis.xlsx

🎯 情感分析统计:
立场分布:
  Neutral: 131 条 (65.2%)
  Opposition: 26 条 (12.9%)
  Support: 26 条 (12.9%)
  Question: 14 条 (7.0%)
  Promotion: 4 条 (2.0%)

情感分布:
  Neutral: 131 条 (65.2%)
  Negative: 42 条 (20.9%)
  Positive: 27 条 (13.4%)
  High: 1 条 (0.5%)

🎉 全部数据分析完成！结果已保存至: DS_v2_dogwifhat_sentiment_analysis.xlsx


In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json
import re

# === 配置 API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)


def analyze_sentiment_wif_perspective_v2(text):
    """更精确的WIF立场分析 - 区分事实陈述和情感表达"""
    prompt = f"""
You are a precise financial sentiment annotator. Analyze the sentiment specifically towards Dogwifhat ($WIF) in the following comment.

Comment: {text}

IMPORTANT:
1. Focus ONLY on sentiment towards Dogwifhat ($WIF)
2. Distinguish between factual statements and emotional opinions
3. If the comment is purely factual/informational without emotional bias, classify as Neutral

Label according to these rules:
- stance ∈ ["Promotion", "Support", "Neutral", "Question", "Opposition"] - specifically towards WIF
- valence ∈ ["Positive", "Neutral", "Negative"] - specifically towards WIF
- arousal ∈ ["Low", "Medium", "High"] - emotional intensity towards WIF
- reason: 1 short sentence explaining the classification

CRITICAL GUIDELINES:
- "Neutral" stance: for factual reports, probability statements, news sharing without opinion
- "Question" stance: for inquiries seeking information
- "Support/Opposition": must contain clear emotional bias or opinion

Output format (strictly valid JSON):
{{"stance": "...", "valence": "...", "arousal": "...", "reason": "..."}}
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": """You are an expert financial sentiment analyst.
                 Analyze sentiment specifically towards Dogwifhat (WIF).
                 CRITICAL: Classify as Neutral for factual statements without emotional bias.
                 Distinguish carefully between information sharing and opinion expression."""},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        result_text = response.choices[0].message.content.strip()
        result_json = json.loads(result_text)
        return result_json

    except Exception as e:
        print(f"❌ API调用错误: {e}")
        return {"stance": "Error", "valence": "Error", "arousal": "Error", "reason": f"API Error: {str(e)}"}

# === 测试修正后的分析 ===
print("🔍 测试修正后的WIF立场分析...")
print("=" * 60)

test_count = 0
results = []

for i, row in df.iterrows():
    text = row['cleaned_text']

    if not text:
        continue

    test_count += 1
    if test_count > 10:
        break

    print(f"\n📝 测试 {test_count}:")
    print(f"原文: {text}")

    result = analyze_sentiment_wif_perspective(text)

    print(f"✅ WIF立场分析结果:")
    print(f"   立场: {result.get('stance')}")
    print(f"   情感: {result.get('valence')}")
    print(f"   强度: {result.get('arousal')}")
    print(f"   理由: {result.get('reason')}")

    results.append(result)
    time.sleep(1.2)

print("\n" + "=" * 60)
print(f"🎯 修正测试完成! 成功分析 {len([r for r in results if 'error' not in r])}/5 条数据")

# 显示WIF立场统计
if results:
    stances = [r.get('stance', 'Unknown') for r in results if 'error' not in r]
    valence = [r.get('valence', 'Unknown') for r in results if 'error' not in r]

    print(f"\n📊 WIF立场统计:")
    print(f"立场分布: {pd.Series(stances).value_counts().to_dict()}")
    print(f"情感分布: {pd.Series(valence).value_counts().to_dict()}")

# === 完整批量分析函数 ===
def batch_analyze_wif_sentiment(df, start_index=0, end_index=None):
    """批量分析对WIF的情绪"""
    if end_index is None:
        end_index = len(df)

    results = []
    valid_texts = df['cleaned_text'].dropna()

    print(f"🚀 开始批量分析WIF立场情绪...")
    print(f"📊 总数据量: {len(valid_texts)}")

    for i, row in df.iterrows():
        if i < start_index:
            continue
        if i >= end_index:
            break

        text = row['cleaned_text']
        if not text:
            results.append({"stance": "Empty", "valence": "Empty", "arousal": "Empty", "reason": "Empty text"})
            continue

        print(f"🔍 分析 {i+1}/{len(df)}: {text[:80]}...")

        try:
            output = analyze_sentiment_wif_perspective(text)
            results.append(output)
            print(f"✅ WIF立场: {output.get('stance', 'N/A')} | 情感: {output.get('valence', 'N/A')}")

        except Exception as e:
            print(f"❌ 分析失败 {i+1}: {e}")
            results.append({"stance": "Error", "valence": "Error", "arousal": "Error", "reason": str(e)})

        time.sleep(1.2)

    return results

# 如果需要运行完整分析，取消注释下面这行
final_results = batch_analyze_wif_sentiment(df)

In [ ]:
# 在分析前可以添加预处理逻辑
def should_be_neutral(text):
    """严格判断是否真正中性"""
    neutral_indicators = [
        "trading at", "price is", "volume was", "market cap",  # 纯数据
        "located at", "founded in", "based in"  # 客观事实
    ]

    non_neutral_indicators = [
        "should", "could", "might", "would",  # 不确定性
        "better than", "worse than", "before", "after",  # 比较
        "hope", "believe", "think", "expect",  # 主观观点
        "unfortunately", "luckily", "sadly"  # 情感词汇
    ]

    text_lower = text.lower()

    # 如果有任何非中性指标，就不是Neutral
    for indicator in non_neutral_indicators:
        if indicator in text_lower:
            return False

    # 只有纯数据/事实才是Neutral
    has_data = any(indicator in text_lower for indicator in neutral_indicators)
    return has_data and len(text.split()) < 15  # 短文本+纯数据

In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json
import re

# === 配置 API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)


def process_financial_content(df):
    """分层处理金融内容"""
    results = []

    for text in df['cleaned_text']:
        # 第一步：内容类型分类
        content_type = classify_content_type(text)

        # 第二步：只有观点类内容才做情感分析
        if content_type in ["Opinion_Expression", "Promotional_Content"]:
            sentiment = analyze_sentiment_only_for_opinions(text)
            results.append({
                "content_type": content_type,
                "sentiment": sentiment,
                "should_include_in_sentiment_analysis": True
            })
        else:
            results.append({
                "content_type": content_type,
                "sentiment": "NotApplicable",
                "should_include_in_sentiment_analysis": False
            })

    return results

In [ ]:
def analyze_wif_content(text):
    """WIF内容分类分析 - 区分事实报道 vs 情感表达"""
    prompt = f"""
You are a cryptocurrency content classifier focused on Dogwifhat ($WIF). Categorize the following comment:

Comment: {text}

Step 1️⃣ — Identify the content type:
- "Factual": Objective statement or report with no opinion or prediction.
- "Opinion": Contains subjective evaluation, attitude, or prediction.
- "Question": Expresses doubt, curiosity, or uncertainty.

Step 2️⃣ — If content is an opinion, classify its stance toward WIF:
["Promotion", "Support", "Neutral", "Opposition"]

Step 3️⃣ — Assign sentiment polarity and intensity:
- valence ∈ ["Positive", "Neutral", "Negative"]
- arousal ∈ ["Low", "Medium", "High"]

Step 4️⃣ — Provide a concise rationale (max 1 sentence) explaining your judgment.

Return strictly in JSON format:
{{
  "content_type": "Factual|Opinion|Question",
  "stance": "Promotion|Support|Neutral|Opposition|NotApplicable",
  "valence": "Positive|Neutral|Negative|NotApplicable",
  "arousal": "Low|Medium|High|NotApplicable",
  "reason": "..."
}}
    """

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are a WIF-focused content classifier. Always focus on sentiment towards Dogwifhat specifically. Distinguish carefully between factual reports and opinion expressions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        result_text = response.choices[0].message.content.strip()
        return json.loads(result_text)

    except Exception as e:
        return {
            "wif_content_type": "Error",
            "wif_sentiment": {"stance": "Error", "intensity": "Error", "confidence": "Error"},
            "reason": str(e)
        }

In [ ]:
# === 2. 读取数据 & 批量调用 ===
df = pd.read_excel("../data/raw/RAW_dogwifhat.xlsx")  # 修改路径
TEXT_COLUMN = "text"

results = []
for i, row in df.iterrows():
    text = str(row[TEXT_COLUMN])
    if not text.strip():
        results.append({
            "content_type": None,
            "stance": None,
            "valence": None,
            "arousal": None,
            "reason": None
        })
        continue

    print(f"Analyzing {i+1}/{len(df)} ...")
    res = analyze_wif_content(text)
    results.append(res)
    time.sleep(1.0)  # 避免触发速率限制

# === 3. 保存结果 ===
result_df = pd.concat([df, pd.DataFrame(results)], axis=1)
result_df.to_excel("dogwifhat_sentiment_multilayer.xlsx", index=False)
print("✅ Finished! Results saved to dogwifhat_sentiment_multilayer.xlsx")

Analyzing 1/201 ...
Analyzing 2/201 ...
Analyzing 3/201 ...
Analyzing 4/201 ...
Analyzing 5/201 ...
Analyzing 6/201 ...
Analyzing 7/201 ...
Analyzing 8/201 ...
Analyzing 9/201 ...
Analyzing 10/201 ...
Analyzing 11/201 ...
Analyzing 12/201 ...
Analyzing 13/201 ...
Analyzing 14/201 ...
Analyzing 15/201 ...
Analyzing 16/201 ...
Analyzing 17/201 ...
Analyzing 18/201 ...
Analyzing 19/201 ...
Analyzing 20/201 ...
Analyzing 21/201 ...
Analyzing 22/201 ...
Analyzing 23/201 ...
Analyzing 24/201 ...
Analyzing 25/201 ...
Analyzing 26/201 ...
Analyzing 27/201 ...
Analyzing 28/201 ...
Analyzing 29/201 ...
Analyzing 30/201 ...
Analyzing 31/201 ...
Analyzing 32/201 ...
Analyzing 33/201 ...
Analyzing 34/201 ...
Analyzing 35/201 ...
Analyzing 36/201 ...
Analyzing 37/201 ...
Analyzing 38/201 ...
Analyzing 39/201 ...
Analyzing 40/201 ...
Analyzing 41/201 ...
Analyzing 42/201 ...
Analyzing 43/201 ...
Analyzing 44/201 ...
Analyzing 45/201 ...
Analyzing 46/201 ...
Analyzing 47/201 ...
Analyzing 48/201 ...
A

In [ ]:
import os
import pandas as pd
from openai import OpenAI
import time
import json

# === 配置 DeepSeek API ===
# set DEEPSEEK_API_KEY in your environment before running
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)

# === 1. 定义情绪分析函数 ===
def analyze_wif_content(text):
    prompt = f"""
You are a financial sentiment analyst specialized in cryptocurrency discourse, focusing on Dogwifhat ($WIF).

Your task is **pure sentiment analysis** — not stance or topic classification.

Step 1️⃣ — Determine if the comment contains any emotional expression toward WIF:
- "HasEmotion": Expresses a clear feeling, attitude, or affect (positive, negative, or otherwise).
- "NoEmotion": Purely factual, descriptive, or informational with no emotional tone.

Step 2️⃣ — If HasEmotion, classify:
- valence ∈ ["Positive", "Neutral", "Negative"]
- arousal ∈ ["Low", "Medium", "High"]

Step 3️⃣ — Provide a brief rationale (max 1 sentence) explaining your reasoning.

Return valid JSON only:
{{
  "emotion_presence": "HasEmotion|NoEmotion",
  "valence": "Positive|Neutral|Negative|NotApplicable",
  "arousal": "Low|Medium|High|NotApplicable",
  "reason": "..."
}}
Comment: {text}
"""
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are an expert crypto sentiment analyst. Focus specifically on emotional tone toward WIF, distinguishing emotion vs. factual content."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
        )

        result_text = response.choices[0].message.content.strip()

        # 尝试提取 JSON
        json_str = result_text[result_text.find("{"): result_text.rfind("}")+1]
        result = json.loads(json_str)

        # 添加可量化分值（方便后续分析）
        valence_map = {"Positive": 1, "Neutral": 0, "Negative": -1, "NotApplicable": None}
        arousal_map = {"Low": 1, "Medium": 2, "High": 3, "NotApplicable": None}

        result["valence_score"] = valence_map.get(result.get("valence"))
        result["arousal_score"] = arousal_map.get(result.get("arousal"))

        return result

    except Exception as e:
        return {
            "emotion_presence": "Error",
            "valence": "Error",
            "arousal": "Error",
            "reason": str(e),
            "valence_score": None,
            "arousal_score": None
        }

# === 2. 读取数据 & 批量分析 ===
df = pd.read_excel("../data/raw/RAW_dogwifhat.xlsx")  # 修改路径
TEXT_COLUMN = "text"

results = []
for i, row in df.iterrows():
    text = str(row[TEXT_COLUMN])
    if not text.strip():
        results.append({
            "emotion_presence": None,
            "valence": None,
            "arousal": None,
            "reason": None,
            "valence_score": None,
            "arousal_score": None
        })
        continue

    print(f"Analyzing {i+1}/{len(df)} ...")
    res = analyze_wif_content(text)
    results.append(res)
    time.sleep(1.0)  # 防止触发速率限制

# === 3. 合并并保存结果 ===
result_df = pd.concat([df, pd.DataFrame(results)], axis=1)
output_path = "dogwifhat_sentiment_multilayer.xlsx"
result_df.to_excel(output_path, index=False)

print(f"✅ Sentiment analysis completed and saved to {output_path}")


Analyzing 1/201 ...
Analyzing 2/201 ...
Analyzing 3/201 ...
Analyzing 4/201 ...
Analyzing 5/201 ...
Analyzing 6/201 ...
Analyzing 7/201 ...
Analyzing 8/201 ...
Analyzing 9/201 ...
Analyzing 10/201 ...
Analyzing 11/201 ...
Analyzing 12/201 ...
Analyzing 13/201 ...
Analyzing 14/201 ...
Analyzing 15/201 ...
Analyzing 16/201 ...
Analyzing 17/201 ...
Analyzing 18/201 ...
Analyzing 19/201 ...
Analyzing 20/201 ...
Analyzing 21/201 ...
Analyzing 22/201 ...
Analyzing 23/201 ...
Analyzing 24/201 ...
Analyzing 25/201 ...
Analyzing 26/201 ...
Analyzing 27/201 ...
Analyzing 28/201 ...
Analyzing 29/201 ...
Analyzing 30/201 ...
Analyzing 31/201 ...
Analyzing 32/201 ...
Analyzing 33/201 ...
Analyzing 34/201 ...
Analyzing 35/201 ...
Analyzing 36/201 ...
Analyzing 37/201 ...
Analyzing 38/201 ...
Analyzing 39/201 ...
Analyzing 40/201 ...
Analyzing 41/201 ...
Analyzing 42/201 ...
Analyzing 43/201 ...
Analyzing 44/201 ...
Analyzing 45/201 ...
Analyzing 46/201 ...
Analyzing 47/201 ...
Analyzing 48/201 ...
A